In [2]:
import pandas as pd
import sqlite3

# Ricarico il dataset
df = pd.read_csv('dati/cubi_STATPOP_01.csv', sep=';', encoding='latin-1')

# Creo il database e ci metto dentro la tabella
conn = sqlite3.connect('demografia.db')
df.to_sql('statpop', conn, if_exists='replace', index=False)

# Verifica
pd.read_sql("SELECT COUNT(*) AS righe FROM statpop", conn)

,righe
0,819720


## 1. I dieci anni con il saldo naturale peggiore
Filtro sul Ticino intero e ordinamento crescente: gli anni in cui il Ticino ha registrato il maggior deficit di nascite.

In [3]:
q1 = """
SELECT Anno, Popolazione AS saldo_naturale
FROM statpop
WHERE Cantone_Distretto_Comune = 'Ticino'
  AND Sesso = 'Sesso - totale'
  AND Nazionalità = 'Nazionalità (categoria) - totale'
  AND Statistiche = 'Incremento naturale'
ORDER BY saldo_naturale ASC
LIMIT 10;
"""
pd.read_sql(q1, conn)

,Anno,saldo_naturale
0,2020,-1561
1,2024,-1117
2,2022,-1102
3,2023,-1098
4,2019,-744
5,2018,-596
6,2021,-562
7,2017,-456
8,2015,-337
9,1981,-228


## 2. Totale e media annua delle due componenti
Aggregazione con SUM e AVG: quanto ha contribuito ciascuna componente in 44 anni.

In [4]:
q2 = """
SELECT Statistiche,
       SUM(Popolazione) AS totale_44_anni,
       ROUND(AVG(Popolazione), 1) AS media_annua,
       MIN(Popolazione) AS peggior_anno,
       MAX(Popolazione) AS miglior_anno
FROM statpop
WHERE Cantone_Distretto_Comune = 'Ticino'
  AND Sesso = 'Sesso - totale'
  AND Nazionalità = 'Nazionalità (categoria) - totale'
  AND Statistiche IN ('Incremento naturale',
                      'Saldo migratorio inclusi i cambiamenti del tipo di popolazione')
GROUP BY Statistiche;
"""
pd.read_sql(q2, conn)

,Statistiche,totale_44_anni,media_annua,peggior_anno,miglior_anno
0,Incremento naturale,-5692,-129.4,-1561,431
1,Saldo migratorio inclusi i cambiamenti del tip...,104038,2364.5,-900,5199


## 3. I comuni con il maggior deficit di nascite
GROUP BY sui singoli comuni, con HAVING per escludere quelli con serie storiche incomplete.

In [5]:
q3 = """
SELECT Cantone_Distretto_Comune AS comune,
       SUM(Popolazione) AS saldo_naturale_cumulato,
       COUNT(*) AS anni_disponibili
FROM statpop
WHERE Statistiche = 'Incremento naturale'
  AND Sesso = 'Sesso - totale'
  AND Nazionalità = 'Nazionalità (categoria) - totale'
  AND Cantone_Distretto_Comune LIKE '5%'
GROUP BY comune
HAVING COUNT(*) >= 40
ORDER BY saldo_naturale_cumulato ASC
LIMIT 15;
"""
pd.read_sql(q3, conn)

,comune,saldo_naturale_cumulato,anni_disponibili
0,5192 Lugano,-1968,44
1,5250 Chiasso,-1253,44
2,5091 Ascona,-1247,44
3,5113 Locarno,-1037,44
4,5118 Minusio,-923,44
5,5120 Muralto,-889,44
6,5254 Mendrisio,-809,44
7,5072 Faido,-551,44
8,5097 Brissago,-526,44
9,5239 Tresa,-489,44


## 4. Variazione anno su anno (window function)
LAG() permette di confrontare ogni riga con la precedente senza fare join della tabella con se stessa.

In [6]:
q4 = """
SELECT Anno,
       Popolazione AS saldo_naturale,
       LAG(Popolazione) OVER (ORDER BY Anno) AS anno_precedente,
       Popolazione - LAG(Popolazione) OVER (ORDER BY Anno) AS variazione
FROM statpop
WHERE Cantone_Distretto_Comune = 'Ticino'
  AND Sesso = 'Sesso - totale'
  AND Nazionalità = 'Nazionalità (categoria) - totale'
  AND Statistiche = 'Incremento naturale'
ORDER BY Anno DESC
LIMIT 15;
"""
pd.read_sql(q4, conn)

,Anno,saldo_naturale,anno_precedente,variazione
0,2024,-1117,-1098,-19
1,2023,-1098,-1102,4
2,2022,-1102,-562,-540
3,2021,-562,-1561,999
4,2020,-1561,-744,-817
5,2019,-744,-596,-148
6,2018,-596,-456,-140
7,2017,-456,-182,-274
8,2016,-182,-337,155
9,2015,-337,-16,-321
